This script makes docker image, pushes to ECR, and makes a Sagemaker Endpoint instance(s)

In [1]:
#import custom sagemaker deploy tools
import sys
sys.path.append('/home/ec2-user/SageMaker/[REDACTED]AppliedScience/Omar/ModelImplementationWSDK/tools')
from sagemaker_deploy_tools import *

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "json" in "MonitoringDatasetFormat" shadows an attribute in parent "Base"
  warnings.warn(


[01/19/25 21:24:45] INFO     Found credentials from IAM Role:                                   ]8;id=390860;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=417961;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py#1075\1075]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


[01/19/25 21:24:46] INFO     Found credentials from IAM Role:                                   ]8;id=63235;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=933707;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py#1075\1075]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

In [2]:
#SET VARIABLES FOR PROJECT HERE

import time
import os
import json

#SET VARIABLES HERE for ECR, model, endpoint, project name
ecr_repository = "[REDACTED]-ml/ela-proficiency-model"
endpoint_name = '[REDACTED]'
instance_type = 'ml.c5.large'

#aws account vairables
account_id = '[REDACTED]'
region = 'us-east-1'


#scaling parameters for server
min_capacity = 1
max_capacity = 100
target_value = 15  # 55/ Target invocations per instance per minute
scale_in_cooldown = 60  # 5 minutes
scale_out_cooldown = 300  # 5 minutes


#subdirectory model + artifacts live in//current folder name
import os
project_name = os.path.basename(os.getcwd())

# Set variables for Sagemaker Endpoints
image_tag = str(round(time.time()*1000))

#copy variables for docker scripts
AWS_ACCOUNT_ID = account_id
REGION = region
IMAGE_TAG = image_tag
ECR_REPOSITORY = ecr_repository


# Construct the ECR image URI
image_uri = f'{account_id}.dkr.ecr.{region}.amazonaws.com/{ecr_repository}:{image_tag}'

print('project name: ', project_name, '\nECR Image URI: ', image_uri)

project name:  ELA_student_proficiency_model 
ECR Image URI:  [REDACTED].dkr.ecr.us-east-1.amazonaws.com/[REDACTED]-ml/ela-proficiency-model:1737321886461


In [3]:
#test message (for inference testing)
test_message = {
      "skillId": "ubuipdtp",
      "questionId": "question_478",
      "eventTime": "2024-10-03T06:20:45.377160",
      "questionIdsHistory": [
        "question_442",
        "question_801",
        "question_633",
        "question_616"
      ],
      "correctnessHistory": [
        100,
        100,
        100,
        0,
        0,
        0,
        100,
        0,
        0
      ],
      "durationSecondsHistory": [
        75,
        217,
        54,
        241,
        97,
        113,
        58,
        32,
        54
      ],
      "eventTimesHistory": [
        "2024-10-03T06:20:45.377160",
        "2024-10-03T06:10:45.377160",
        "2024-10-03T06:00:45.377160",
        "2024-10-03T05:50:45.377160",
        "2024-10-03T05:40:45.377160",
        "2024-10-03T05:30:45.377160",
        "2024-10-03T05:20:45.377160",
        "2024-10-03T05:10:45.377160",
        "2024-10-03T05:00:45.377160"
      ]
    }


In [4]:
#MODIFY YOUR DOCKERFILE TO INCLUDE ALL ARTIFACTS NEEDED
#Check your library depedencies



In [5]:
#buid docker, push to ECR

import subprocess

def run_command(command):
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, shell=True)
    output, error = process.communicate()
    if process.returncode != 0:
        print(f"Error executing command: {command}")
        print(f"Error message: {error.decode('utf-8')}")
        raise Exception(f"Command failed with return code {process.returncode}")
    return output.decode('utf-8')

#look at dockerfile
dockerfile_path = os.path.join('container', 'Dockerfile')
with open(dockerfile_path, 'r') as file:
    dockerfile_contents = file.read()
print("Contents of container/Dockerfile:")
print(dockerfile_contents)

# Change directory to 'container'
os.chdir('container')

# Authenticate Docker to AWS ECR
run_command(f"aws ecr get-login-password --region {REGION} | docker login --username AWS --password-stdin 763104351884.dkr.ecr.{REGION}.amazonaws.com")

# Also authenticate to your own ECR repository
run_command(f"aws ecr get-login-password --region {REGION} | docker login --username AWS --password-stdin {AWS_ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com")

# Create ECR repository if it doesn't exist
try:
    run_command(f"aws ecr describe-repositories --repository-names {ECR_REPOSITORY}")
except Exception:
    run_command(f"aws ecr create-repository --repository-name {ECR_REPOSITORY}")

# Build the Docker image
run_command(f"docker build -t {ECR_REPOSITORY}:{IMAGE_TAG} .")

# Tag the image for ECR
run_command(f"docker tag {ECR_REPOSITORY}:{IMAGE_TAG} {AWS_ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/{ECR_REPOSITORY}:{IMAGE_TAG}")


Contents of container/Dockerfile:
# Start with the AWS Deep Learning Container for Python 3.8
#FROM 763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-inference:1.8.1-cpu-py38-ubuntu20.04
# FROM pytorch/pytorch:1.8.1-cuda11.1-cudnn8-runtime
FROM 763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-inference:1.12.1-cpu-py38-ubuntu20.04-sagemaker

# Set working directory
WORKDIR /opt/ml/model

# Install the required packages
RUN pip install --no-cache-dir --upgrade pip && \
    pip install --no-cache-dir \
    numpy \
    scipy \
    pandas \
    flask \
    gunicorn \
    pickle5 
RUN pip install --no-cache-dir -v xgboost==2.1.1
RUN pip install --no-cache-dir joblib==1.4.2
RUN pip install --no-cache-dir scikit-learn
#RUN pip install --no-cache-dir numpy==1.26.4
#RUN pip install --no-cache-dir pandas==1.5.3

# Copy your model file and any other necessary files
COPY proficiency_model.json /opt/ml/model/
#COPY proficiency_scaler.json /opt/ml/model/
COPY confidence_score_scaler.json /opt/

In [6]:
#Runs local docker image test before deploy

import requests
import json

def test_docker_inference(repository, tag, test_message):
    try:
        # Run the container
        run_command(f"docker run -d -p 8080:8080 --name test-container {repository}:{tag}")
        print("Container started, waiting for it to be ready...")
        
        import time
        time.sleep(5)
        
        # Test inference request
        inference_response = requests.post(
            'http://localhost:8080/invocations',
            json=test_message,
            headers={'Content-Type': 'application/json'}
        )
        
        print(f"Inference response status: {inference_response.status_code}")
        if inference_response.status_code == 200:
            print("Response:", inference_response.json())
            return True  # Successfully tested
        else:
            raise Exception(f"Test failed with status {inference_response.status_code}: {inference_response.text}")
        
    except Exception as e:
        print(f"Error testing container: {str(e)}")
        raise  # Re-raise the exception to stop notebook execution
    finally:
        run_command("docker stop test-container")
        run_command("docker rm test-container")
        print('cleaned up/removed docker container')
        
# Add after build but before deploy:
test_docker_inference(ECR_REPOSITORY, IMAGE_TAG, test_message)

Container started, waiting for it to be ready...
Inference response status: 200
Response: {'debug_info': {'parsed_json': {'correctnessHistory': [100, 100, 100, 0, 0, 0, 100, 0, 0], 'durationSecondsHistory': [75, 217, 54, 241, 97, 113, 58, 32, 54], 'eventTime': '2024-10-03T06:20:45.377160', 'eventTimesHistory': ['2024-10-03T06:20:45.377160', '2024-10-03T06:10:45.377160', '2024-10-03T06:00:45.377160', '2024-10-03T05:50:45.377160', '2024-10-03T05:40:45.377160', '2024-10-03T05:30:45.377160', '2024-10-03T05:20:45.377160', '2024-10-03T05:10:45.377160', '2024-10-03T05:00:45.377160'], 'questionId': 'question_478', 'questionIdsHistory': ['question_442', 'question_801', 'question_633', 'question_616'], 'skillId': 'ubuipdtp'}, 'parsed_json_length': 603, 'received_content_type': 'application/json', 'received_data_length': 603, 'received_raw_data': '{"skillId": "ubuipdtp", "questionId": "question_478", "eventTime": "2024-10-03T06:20:45.377160", "questionIdsHistory": ["question_442", "question_801",

True

In [ ]:
# Push the image to ECR
run_command(f"docker push {AWS_ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/{ECR_REPOSITORY}:{IMAGE_TAG}")

print(f"Image pushed successfully to ECR: {AWS_ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/{ECR_REPOSITORY}:{IMAGE_TAG}")

In [7]:
#Deploy if it doesn't exist; otherwise rolling update; rollback if fails

import boto3
import sagemaker
from sagemaker import get_execution_role
from botocore.exceptions import ClientError

# Initialize the SageMaker client
sagemaker_client = boto3.client('sagemaker')

# Set up the SageMaker session and role
sagemaker_session = sagemaker.Session()
role = get_execution_role()


# Create the model
model = sagemaker.Model(
    image_uri=image_uri,
    role=role
)




# Check if the endpoint exists
if not endpoint_exists(endpoint_name):
    # Deploy the model to create an endpoint
    predictor = model.deploy(
        initial_instance_count=1,
        instance_type=instance_type,
        endpoint_name=endpoint_name
    )

    #Create endpoint
    print(f"Endpoint '{endpoint_name}' is being created. This may take a few minutes...")

    # Wait for the endpoint to be in service
    waiter = boto3.client('sagemaker').get_waiter('endpoint_in_service')
    waiter.wait(EndpointName=endpoint_name)

    print(f"Endpoint '{endpoint_name}' is now in service and ready for real-time inference.")
    
    print(f"Attempting to register scaling.")    
    setup_auto_scaling(endpoint_name, min_capacity, max_capacity, target_value, 
                       scale_in_cooldown, scale_out_cooldown)
    
    print(f"Finished.")
else:
    #Endpoint exists, do a rolling update/deploy
    print(f"Endpoint '{endpoint_name}' already exists.")
    print('Attempting update...')
    #this function will descale, update, then rescale
    update_endpoint_with_scaling(endpoint_name, image_uri, test_message, min_capacity, max_capacity, target_value, scale_in_cooldown, scale_out_cooldown)
    

Endpoint '[REDACTED]' already exists.
Attempting update...
Current endpoint status: InService
Attempting deregistration of scalable target...
Scalable target for endpoint [REDACTED] not found. Proceeding with update.
Attempting updating of endpoint...
Current Endpoint Config: ela-20250103042937
Current Model Name: ela-proficiency-model-2025-01-03-04-29-36-636
Current Image URI: [REDACTED].dkr.ecr.us-east-1.amazonaws.com/[REDACTED]-ml/ela-proficiency-model:1735878570474
New config name:  ela-20250119212457  Old config name:  ela-20250103042937
Endpoint '[REDACTED]' is being updated. This may take a few minutes...
Endpoint '[REDACTED]' has been successfully updated with the new model.
Attempting registration of scaling of target...
No existing scaling policy found
No existing scalable target found
Auto-scaling has been set up for endpoint: [REDACTED]
Min capacity: 1
Max capacity: 100
Target value: 15 invocations per instance per minute
Scale-in cooldown: 60 seconds
Scale-out cooldown: 30

Unit Tests (bunch of test messages)

In [8]:
#test endpoint
import boto3
import json

runtime = boto3.client('sagemaker-runtime')
print('submitting: ', test_message)
response = runtime.invoke_endpoint(EndpointName=endpoint_name, 
                                   ContentType='application/json', 
                                   Body=json.dumps(test_message))
result = json.loads(response['Body'].read().decode())

print('result: ', result)

submitting:  {'skillId': 'ubuipdtp', 'questionId': 'question_478', 'eventTime': '2024-10-03T06:20:45.377160', 'questionIdsHistory': ['question_442', 'question_801', 'question_633', 'question_616'], 'correctnessHistory': [100, 100, 100, 0, 0, 0, 100, 0, 0], 'durationSecondsHistory': [75, 217, 54, 241, 97, 113, 58, 32, 54], 'eventTimesHistory': ['2024-10-03T06:20:45.377160', '2024-10-03T06:10:45.377160', '2024-10-03T06:00:45.377160', '2024-10-03T05:50:45.377160', '2024-10-03T05:40:45.377160', '2024-10-03T05:30:45.377160', '2024-10-03T05:20:45.377160', '2024-10-03T05:10:45.377160', '2024-10-03T05:00:45.377160']}
result:  {'debug_info': {'parsed_json': {'correctnessHistory': [100, 100, 100, 0, 0, 0, 100, 0, 0], 'durationSecondsHistory': [75, 217, 54, 241, 97, 113, 58, 32, 54], 'eventTime': '2024-10-03T06:20:45.377160', 'eventTimesHistory': ['2024-10-03T06:20:45.377160', '2024-10-03T06:10:45.377160', '2024-10-03T06:00:45.377160', '2024-10-03T05:50:45.377160', '2024-10-03T05:40:45.377160', '

In [9]:
#test endpoint with double json-ifciation

try:

    runtime = boto3.client('sagemaker-runtime')
    print('submitting: ', test_message)
    response = runtime.invoke_endpoint(EndpointName=endpoint_name, 
                                       ContentType='application/json', 
                                       Body=json.dumps(str(json.dumps(test_message))))
    result = json.loads(response['Body'].read().decode())

    print('unsuccessfully failed.')
    print('result: ', result)
    
except Exception as e:
    print('successfully failed.')
    print('error: ', e)

submitting:  {'skillId': 'ubuipdtp', 'questionId': 'question_478', 'eventTime': '2024-10-03T06:20:45.377160', 'questionIdsHistory': ['question_442', 'question_801', 'question_633', 'question_616'], 'correctnessHistory': [100, 100, 100, 0, 0, 0, 100, 0, 0], 'durationSecondsHistory': [75, 217, 54, 241, 97, 113, 58, 32, 54], 'eventTimesHistory': ['2024-10-03T06:20:45.377160', '2024-10-03T06:10:45.377160', '2024-10-03T06:00:45.377160', '2024-10-03T05:50:45.377160', '2024-10-03T05:40:45.377160', '2024-10-03T05:30:45.377160', '2024-10-03T05:20:45.377160', '2024-10-03T05:10:45.377160', '2024-10-03T05:00:45.377160']}
successfully failed.
error:  An error occurred (ModelError) when calling the InvokeEndpoint operation: Received server error (500) from primary and could not load the entire response body. See https://us-east-1.console.aws.amazon.com/cloudwatch/home?region=us-east-1#logEventViewer:group=/aws/sagemaker/Endpoints/[REDACTED] in account [REDACTED] for more information.


In [10]:
#test endpoint with non json-ifciation
import boto3
import json

try:
    runtime = boto3.client('sagemaker-runtime')
    print('submitting: ', test_message)
    response = runtime.invoke_endpoint(EndpointName=endpoint_name, 
                                       ContentType='application/json', 
                                       Body=test_message)
    result = json.loads(response['Body'].read().decode())

    print('result: ', result)
except Exception as e:
    print('successfully failed.')
    print('error: ', e)

submitting:  {'skillId': 'ubuipdtp', 'questionId': 'question_478', 'eventTime': '2024-10-03T06:20:45.377160', 'questionIdsHistory': ['question_442', 'question_801', 'question_633', 'question_616'], 'correctnessHistory': [100, 100, 100, 0, 0, 0, 100, 0, 0], 'durationSecondsHistory': [75, 217, 54, 241, 97, 113, 58, 32, 54], 'eventTimesHistory': ['2024-10-03T06:20:45.377160', '2024-10-03T06:10:45.377160', '2024-10-03T06:00:45.377160', '2024-10-03T05:50:45.377160', '2024-10-03T05:40:45.377160', '2024-10-03T05:30:45.377160', '2024-10-03T05:20:45.377160', '2024-10-03T05:10:45.377160', '2024-10-03T05:00:45.377160']}
successfully failed.
error:  Parameter validation failed:
Invalid type for parameter Body, value: {'skillId': 'ubuipdtp', 'questionId': 'question_478', 'eventTime': '2024-10-03T06:20:45.377160', 'questionIdsHistory': ['question_442', 'question_801', 'question_633', 'question_616'], 'correctnessHistory': [100, 100, 100, 0, 0, 0, 100, 0, 0], 'durationSecondsHistory': [75, 217, 54, 2

In [11]:
test_message_multiple_events = [{'skillId': '5111f61e-e69f-e311-9503-005056801da1',
  'questionId': 'question_136',
  'eventTime': '2024-10-04T19:15:18.187471',
  'questionIdsHistory': ['question_852',
   'question_204',
   'question_75',
   'question_862',
   'question_775',
   'question_649',
   'question_928',
   'question_242',
   'question_322'],
  'correctnessHistory': [100, 100, 0, 0, 0, 0, 100, 100, 0],
  'durationSecondsHistory': [142, 51, 245, 214, 242, 147, 18, 16, 85],
  'eventTimesHistory': ['2024-10-04T19:15:18.187471',
   '2024-10-04T19:05:18.187471',
   '2024-10-04T18:55:18.187471',
   '2024-10-04T18:45:18.187471',
   '2024-10-04T18:35:18.187471',
   '2024-10-04T18:25:18.187471',
   '2024-10-04T18:15:18.187471',
   '2024-10-04T18:05:18.187471',
   '2024-10-04T17:55:18.187471']},
 {'skillId': 'wsudnogn',
  'questionId': 'question_785',
  'eventTime': '2024-10-04T19:15:18.187522',
  'questionIdsHistory': ['question_25',
   'question_620',
   'question_255',
   'question_935',
   'question_403',
   'question_760',
   'question_864',
   'question_153',
   'question_585'],
  'correctnessHistory': [100, 100, 0, 0, 100, 0, 0, 0, 100],
  'durationSecondsHistory': [122, 245, 26, 154, 71, 57, 52, 156, 201],
  'eventTimesHistory': ['2024-10-04T19:15:18.187522',
   '2024-10-04T19:05:18.187522',
   '2024-10-04T18:55:18.187522',
   '2024-10-04T18:45:18.187522',
   '2024-10-04T18:35:18.187522',
   '2024-10-04T18:25:18.187522',
   '2024-10-04T18:15:18.187522',
   '2024-10-04T18:05:18.187522',
   '2024-10-04T17:55:18.187522']},
 {'skillId': 'prtvczlp',
  'questionId': 'question_586',
  'eventTime': '2024-10-04T19:15:18.187562',
  'questionIdsHistory': ['question_692',
   'question_238',
   'question_688',
   'question_643',
   'question_4',
   'question_781',
   'question_885',
   'question_594',
   'question_392'],
  'correctnessHistory': [100, 100, 0, 100, 0, 0, 0, 0, 0],
  'durationSecondsHistory': [126, 10, 89, 167, 174, 127, 283, 39, 272],
  'eventTimesHistory': ['2024-10-04T19:15:18.187562',
   '2024-10-04T19:05:18.187562',
   '2024-10-04T18:55:18.187562',
   '2024-10-04T18:45:18.187562',
   '2024-10-04T18:35:18.187562',
   '2024-10-04T18:25:18.187562',
   '2024-10-04T18:15:18.187562',
   '2024-10-04T18:05:18.187562',
   '2024-10-04T17:55:18.187562']},
 {'skillId': 'tnzjudrl',
  'questionId': 'question_917',
  'eventTime': '2024-10-04T19:15:18.187602',
  'questionIdsHistory': ['question_19',
   'question_561',
   'question_720',
   'question_911',
   'question_384',
   'question_803',
   'question_82',
   'question_601',
   'question_778'],
  'correctnessHistory': [100, 0, 0, 0, 0, 0, 100, 0, 0],
  'durationSecondsHistory': [176, 82, 275, 270, 34, 211, 197, 268, 280],
  'eventTimesHistory': ['2024-10-04T19:15:18.187602',
   '2024-10-04T19:05:18.187602',
   '2024-10-04T18:55:18.187602',
   '2024-10-04T18:45:18.187602',
   '2024-10-04T18:35:18.187602',
   '2024-10-04T18:25:18.187602',
   '2024-10-04T18:15:18.187602',
   '2024-10-04T18:05:18.187602',
   '2024-10-04T17:55:18.187602']},
 {'skillId': 'FAKE_AFVBV',
  'questionId': 'question_477',
  'eventTime': '2024-10-04T19:15:18.187641',
  'questionIdsHistory': ['question_994',
   'question_24',
   'question_59',
   'question_895',
   'question_403',
   'question_662',
   'question_147',
   'question_546',
   'question_69'],
  'correctnessHistory': [100, 0, 100, 0, 0, 0, 0, 100, 0],
  'durationSecondsHistory': [245, 111, 189, 24, 174, 175, 67, 68, 270],
  'eventTimesHistory': ['2024-10-04T19:15:18.187641',
   '2024-10-04T19:05:18.187641',
   '2024-10-04T18:55:18.187641',
   '2024-10-04T18:45:18.187641',
   '2024-10-04T18:35:18.187641',
   '2024-10-04T18:25:18.187641',
   '2024-10-04T18:15:18.187641',
   '2024-10-04T18:05:18.187641',
   '2024-10-04T17:55:18.187641']}]


#test endpoint
import boto3
import json

runtime = boto3.client('sagemaker-runtime')
print('submitting: ', test_message_multiple_events)
response = runtime.invoke_endpoint(EndpointName=endpoint_name, 
                                   ContentType='application/json', 
                                   Body=json.dumps(test_message_multiple_events))
result = json.loads(response['Body'].read().decode())

print('result: ', result)
print('PREDICTION: ', result['prediction'])

submitting:  [{'skillId': '5111f61e-e69f-e311-9503-005056801da1', 'questionId': 'question_136', 'eventTime': '2024-10-04T19:15:18.187471', 'questionIdsHistory': ['question_852', 'question_204', 'question_75', 'question_862', 'question_775', 'question_649', 'question_928', 'question_242', 'question_322'], 'correctnessHistory': [100, 100, 0, 0, 0, 0, 100, 100, 0], 'durationSecondsHistory': [142, 51, 245, 214, 242, 147, 18, 16, 85], 'eventTimesHistory': ['2024-10-04T19:15:18.187471', '2024-10-04T19:05:18.187471', '2024-10-04T18:55:18.187471', '2024-10-04T18:45:18.187471', '2024-10-04T18:35:18.187471', '2024-10-04T18:25:18.187471', '2024-10-04T18:15:18.187471', '2024-10-04T18:05:18.187471', '2024-10-04T17:55:18.187471']}, {'skillId': 'wsudnogn', 'questionId': 'question_785', 'eventTime': '2024-10-04T19:15:18.187522', 'questionIdsHistory': ['question_25', 'question_620', 'question_255', 'question_935', 'question_403', 'question_760', 'question_864', 'question_153', 'question_585'], 'correct

In [12]:
test_message_multiple_events = [{'skillId': '5111f61e-e69f-e311-9503-005056801da1',
  'questionId': 'question_136',
  'eventTime': '2024-10-04T19:15:18.187471',
  'questionIdsHistory': ['question_852',
   'question_204',
   'question_75',
   'question_862',
   'question_775',
   'question_649',
   'question_928',
   'question_242',
   'question_322'],
  'correctnessHistory': [100, 100, 0, 0, 0, 0, 100, 100, 0],
  'durationSecondsHistory': [142, 51, 245, 214, 242, 147, 18, 16, 85],
  'eventTimesHistory': ['2024-10-04T19:15:18.187471',
   '2024-10-04T19:05:18.187471',
   '2024-10-04T18:55:18.187471',
   '2024-10-04T18:45:18.187471',
   '2024-10-04T18:35:18.187471',
   '2024-10-04T18:25:18.187471',
   '2024-10-04T18:15:18.187471',
   '2024-10-04T18:05:18.187471',
   '2024-10-04T17:55:18.187471']},
 {'skillId': 'wsudnogn',
  'questionId': 'question_785',
  'eventTime': '2024-10-04T19:15:18.187522',
  'questionIdsHistory': ['question_25',
   'question_620',
   'question_255',
   'question_935',
   'question_403',
   'question_760',
   'question_864',
   'question_153',
   'question_585'],
  'correctnessHistory': [100, 100, 0, 0, 100, 0, 0, 0, 100],
  'durationSecondsHistory': [122, 245, 26, 154, 71, 57, 52, 156, 201],
  'eventTimesHistory': ['2024-10-04T19:15:18.187522',
   '2024-10-04T19:05:18.187522',
   '2024-10-04T18:55:18.187522',
   '2024-10-04T18:45:18.187522',
   '2024-10-04T18:35:18.187522',
   '2024-10-04T18:25:18.187522',
   '2024-10-04T18:15:18.187522',
   '2024-10-04T18:05:18.187522',
   '2024-10-04T17:55:18.187522']},
 {'skillId': 'prtvczlp',
  'questionId': 'question_586',
  'eventTime': '2024-10-04T19:15:18.187562',
  'questionIdsHistory': ['question_692',
   'question_238',
   'question_688',
   'question_643',
   'question_4',
   'question_781',
   'question_885',
   'question_594',
   'question_392'],
  'correctnessHistory': [100, 100, 0, 100, 0, 0, 0, 0, 0],
  'durationSecondsHistory': [126, 10, 89, 167, 174, 127, 283, 39, 272],
  'eventTimesHistory': ['2024-10-04T19:15:18.187562',
   '2024-10-04T19:05:18.187562',
   '2024-10-04T18:55:18.187562',
   '2024-10-04T18:45:18.187562',
   '2024-10-04T18:35:18.187562',
   '2024-10-04T18:25:18.187562',
   '2024-10-04T18:15:18.187562',
   '2024-10-04T18:05:18.187562',
   '2024-10-04T17:55:18.187562']},
 {'skillId': 'tnzjudrl',
  'questionId': 'question_917',
  'eventTime': '2024-10-04T19:15:18.187602',
  'questionIdsHistory': ['question_19',
   'question_561',
   'question_720',
   'question_911',
   'question_384',
   'question_803',
   'question_82',
   'question_601',
   'question_778'],
  'correctnessHistory': [100, 0, 0, 0, 0, 0, 100, 0, 0],
  'durationSecondsHistory': [],
  'eventTimesHistory': ['2024-10-04T19:15:18.187602',
   '2024-10-04T19:05:18.187602',
   '2024-10-04T18:55:18.187602',
   '2024-10-04T18:45:18.187602',
   '2024-10-04T18:35:18.187602',
   '2024-10-04T18:25:18.187602',
   '2024-10-04T18:15:18.187602',
   '2024-10-04T18:05:18.187602',
   '2024-10-04T17:55:18.187602']},
 {'skillId': 'FAKE_AFVBV',
  'questionId': 'question_477',
  'eventTime': '2024-10-04T19:15:18.187641',
  'questionIdsHistory': ['question_994',
   'question_24',
   'question_59',
   'question_895',
   'question_403',
   'question_662',
   'question_147',
   'question_546',
   'question_69'],
  'correctnessHistory': [100, 0, 100, 0, 0, 0, 0, 100, 0],
  'durationSecondsHistory': [245, 111, 189, 24, 174, 175, 67, 68, 270],
  'eventTimesHistory': ['2024-10-04T19:15:18.187641',
   '2024-10-04T19:05:18.187641',
   '2024-10-04T18:55:18.187641',
   '2024-10-04T18:45:18.187641',
   '2024-10-04T18:35:18.187641',
   '2024-10-04T18:25:18.187641',
   '2024-10-04T18:15:18.187641',
   '2024-10-04T18:05:18.187641',
   '2024-10-04T17:55:18.187641']}]


#test endpoint
import boto3
import json

runtime = boto3.client('sagemaker-runtime')
print('submitting: ', test_message_multiple_events)
response = runtime.invoke_endpoint(EndpointName=endpoint_name, 
                                   ContentType='application/json', 
                                   Body=json.dumps(test_message_multiple_events))
result = json.loads(response['Body'].read().decode())

print('result: ', result)

submitting:  [{'skillId': '5111f61e-e69f-e311-9503-005056801da1', 'questionId': 'question_136', 'eventTime': '2024-10-04T19:15:18.187471', 'questionIdsHistory': ['question_852', 'question_204', 'question_75', 'question_862', 'question_775', 'question_649', 'question_928', 'question_242', 'question_322'], 'correctnessHistory': [100, 100, 0, 0, 0, 0, 100, 100, 0], 'durationSecondsHistory': [142, 51, 245, 214, 242, 147, 18, 16, 85], 'eventTimesHistory': ['2024-10-04T19:15:18.187471', '2024-10-04T19:05:18.187471', '2024-10-04T18:55:18.187471', '2024-10-04T18:45:18.187471', '2024-10-04T18:35:18.187471', '2024-10-04T18:25:18.187471', '2024-10-04T18:15:18.187471', '2024-10-04T18:05:18.187471', '2024-10-04T17:55:18.187471']}, {'skillId': 'wsudnogn', 'questionId': 'question_785', 'eventTime': '2024-10-04T19:15:18.187522', 'questionIdsHistory': ['question_25', 'question_620', 'question_255', 'question_935', 'question_403', 'question_760', 'question_864', 'question_153', 'question_585'], 'correct

In [13]:
result['prediction']

{'item_prediction': [0.44560787081718445,
  0.48265090584754944,
  0.4974043369293213,
  0.36954227089881897,
  0.4533672034740448],
 'item_prediction_confidence': [81.75720992622401,
  26.556502133367584,
  3.8514776602879692,
  4.185396064329238,
  1.3299656092583874],
 'item_prediction_error': [0.46531572937965393,
  0.46861639618873596,
  0.4719783067703247,
  0.45849451422691345,
  0.468679279088974],
 'model_version': ['4.0a5', '4.0a5', '4.0a5', '4.0a5', '4.0a5'],
 'skill_prediction': [0.5907628536224365,
  0.6474894881248474,
  0.6489365696907043,
  0.3617837727069855,
  0.4748389720916748],
 'skill_prediction_confidence': [0.0014270017266720893,
  81.75720992622401,
  20.44179973457768,
  34.19666937796995,
  0.0014270017266720893],
 'skill_prediction_error': [0.5150402784347534,
  0.4737383723258972,
  0.5122877359390259,
  0.45647501945495605,
  0.5201220512390137]}

In [6]:
#test endpoint with empty time message
empty_test_message = {'skillId': 'ubuipdtp', 
                    'questionId': 'question_478', 
                    'eventTime': '',
                    'questionIdsHistory': ['question_442', 'question_801', 'question_633', 'question_616', 
                                           'question_367', 
                                           'question_465', 'question_184', 'question_283', 'question_327'], 
                    'correctnessHistory': [], 
                    'durationSecondsHistory': [], 
                    'eventTimesHistory': []}


import boto3
import json

runtime = boto3.client('sagemaker-runtime')
print('submitting: ', empty_test_message)
response = runtime.invoke_endpoint(EndpointName=endpoint_name, 
                                   ContentType='application/json', 
                                   Body=json.dumps(empty_test_message))
result = json.loads(response['Body'].read().decode())

print('result: ', result)

submitting:  {'skillId': 'ubuipdtp', 'questionId': 'question_478', 'eventTime': '', 'questionIdsHistory': ['question_442', 'question_801', 'question_633', 'question_616', 'question_367', 'question_465', 'question_184', 'question_283', 'question_327'], 'correctnessHistory': [], 'durationSecondsHistory': [], 'eventTimesHistory': []}
result:  {'debug_info': {'parsed_json': {'correctnessHistory': [], 'durationSecondsHistory': [], 'eventTime': '', 'eventTimesHistory': [], 'questionId': 'question_478', 'questionIdsHistory': ['question_442', 'question_801', 'question_633', 'question_616', 'question_367', 'question_465', 'question_184', 'question_283', 'question_327'], 'skillId': 'ubuipdtp'}, 'parsed_json_length': 319, 'received_content_type': 'application/json', 'received_data_length': 319, 'received_raw_data': '{"skillId": "ubuipdtp", "questionId": "question_478", "eventTime": "", "questionIdsHistory": ["question_442", "question_801", "question_633", "question_616", "question_367", "question

In [7]:
#test endpoint with empty qid message & empty event time
empty_test_message = {'skillId': 'ubuipdtp', 
                    'questionId': '', 
                    'eventTime': '',
                    'questionIdsHistory': ['question_442', 'question_801', 'question_633', 'question_616', 
                                           'question_367', 
                                           'question_465', 'question_184', 'question_283', 'question_327'], 
                    'correctnessHistory': [], 
                    'durationSecondsHistory': [], 
                    'eventTimesHistory': []}


import boto3
import json

runtime = boto3.client('sagemaker-runtime')
print('submitting: ', empty_test_message)
response = runtime.invoke_endpoint(EndpointName=endpoint_name, 
                                   ContentType='application/json', 
                                   Body=json.dumps(empty_test_message))
result = json.loads(response['Body'].read().decode())

print('result: ', result)

submitting:  {'skillId': 'ubuipdtp', 'questionId': '', 'eventTime': '', 'questionIdsHistory': ['question_442', 'question_801', 'question_633', 'question_616', 'question_367', 'question_465', 'question_184', 'question_283', 'question_327'], 'correctnessHistory': [], 'durationSecondsHistory': [], 'eventTimesHistory': []}
result:  {'debug_info': {'parsed_json': {'correctnessHistory': [], 'durationSecondsHistory': [], 'eventTime': '', 'eventTimesHistory': [], 'questionId': '', 'questionIdsHistory': ['question_442', 'question_801', 'question_633', 'question_616', 'question_367', 'question_465', 'question_184', 'question_283', 'question_327'], 'skillId': 'ubuipdtp'}, 'parsed_json_length': 307, 'received_content_type': 'application/json', 'received_data_length': 307, 'received_raw_data': '{"skillId": "ubuipdtp", "questionId": "", "eventTime": "", "questionIdsHistory": ["question_442", "question_801", "question_633", "question_616", "question_367", "question_465", "question_184", "question_283

In [15]:
reproduced_test_message = {
    'skillId': 'sqyudjgp',
    'questionId': '6dc12d68-af7f-441f-9839-5882e115a2ca',
    'eventTime': '2024-12-10 13:34:29.371403',
    'questionIdsHistory': [], 
    'correctnessHistory': [],
    'durationSecondsHistory': [],
    'eventTimesHistory': []
}

print('submitting: ', empty_test_message)
response = runtime.invoke_endpoint(EndpointName=endpoint_name, 
                                   ContentType='application/json', 
                                   Body=json.dumps(reproduced_test_message))
result = json.loads(response['Body'].read().decode())

print('result: ', result)

submitting:  {'skillId': 'ubuipdtp', 'questionId': 'question_478', 'eventTime': '', 'questionIdsHistory': ['question_442', 'question_801', 'question_633', 'question_616', 'question_367', 'question_465', 'question_184', 'question_283', 'question_327'], 'correctnessHistory': [], 'durationSecondsHistory': [], 'eventTimesHistory': []}
result:  {'debug_info': {'parsed_json': {'correctnessHistory': [], 'durationSecondsHistory': [], 'eventTime': '2024-12-10 13:34:29.371403', 'eventTimesHistory': [], 'questionId': '6dc12d68-af7f-441f-9839-5882e115a2ca', 'questionIdsHistory': [], 'skillId': 'sqyudjgp'}, 'parsed_json_length': 227, 'received_content_type': 'application/json', 'received_data_length': 227, 'received_raw_data': '{"skillId": "sqyudjgp", "questionId": "6dc12d68-af7f-441f-9839-5882e115a2ca", "eventTime": "2024-12-10 13:34:29.371403", "questionIdsHistory": [], "correctnessHistory": [], "durationSecondsHistory": [], "eventTimesHistory": []}'}, 'prediction': {'item_prediction': [0.6134344

In [16]:
reproduced_test_message = {
    'skillId': 'sqyudjgp',
    'questionId': '6dc12d68-af7f-441f-9839-5882e115a2ca',
    'eventTime': '2024-10-04T19:15:18.187471',
    'questionIdsHistory': ['question_442', 'question_801', 'question_633', 'question_616', 
                           'question_367', 
                           'question_465', 'question_184', 'question_283', 'question_327'], 
    'correctnessHistory': [],
    'durationSecondsHistory': [],
    'eventTimesHistory': []
}

print('submitting: ', empty_test_message)
response = runtime.invoke_endpoint(EndpointName=endpoint_name, 
                                   ContentType='application/json', 
                                   Body=json.dumps(reproduced_test_message))
result = json.loads(response['Body'].read().decode())

print('result: ', result)

submitting:  {'skillId': 'ubuipdtp', 'questionId': 'question_478', 'eventTime': '', 'questionIdsHistory': ['question_442', 'question_801', 'question_633', 'question_616', 'question_367', 'question_465', 'question_184', 'question_283', 'question_327'], 'correctnessHistory': [], 'durationSecondsHistory': [], 'eventTimesHistory': []}
result:  {'debug_info': {'parsed_json': {'correctnessHistory': [], 'durationSecondsHistory': [], 'eventTime': '2024-10-04T19:15:18.187471', 'eventTimesHistory': [], 'questionId': '6dc12d68-af7f-441f-9839-5882e115a2ca', 'questionIdsHistory': ['question_442', 'question_801', 'question_633', 'question_616', 'question_367', 'question_465', 'question_184', 'question_283', 'question_327'], 'skillId': 'sqyudjgp'}, 'parsed_json_length': 369, 'received_content_type': 'application/json', 'received_data_length': 369, 'received_raw_data': '{"skillId": "sqyudjgp", "questionId": "6dc12d68-af7f-441f-9839-5882e115a2ca", "eventTime": "2024-10-04T19:15:18.187471", "questionIds

In [17]:
 reproduced_test_message = {
    'skillId': 'sqyudjgp',
    'questionId': '6dc12d68-af7f-441f-9839-5882e115a2ca',
    'eventTime': '2024-10-04T19:15:18.187471',
    'questionIdsHistory': ['question_442', 'question_801', 'question_633', 'question_616', 
                           'question_465', 'question_184', 'question_283', 'question_327'], 
    'correctnessHistory': [],
    'durationSecondsHistory': [],
    'eventTimesHistory': []
}

print('submitting: ', empty_test_message)
response = runtime.invoke_endpoint(EndpointName=endpoint_name, 
                                   ContentType='application/json', 
                                   Body=json.dumps(reproduced_test_message))
result = json.loads(response['Body'].read().decode())

print('result: ', result)

submitting:  {'skillId': 'ubuipdtp', 'questionId': 'question_478', 'eventTime': '', 'questionIdsHistory': ['question_442', 'question_801', 'question_633', 'question_616', 'question_367', 'question_465', 'question_184', 'question_283', 'question_327'], 'correctnessHistory': [], 'durationSecondsHistory': [], 'eventTimesHistory': []}
result:  {'debug_info': {'parsed_json': {'correctnessHistory': [], 'durationSecondsHistory': [], 'eventTime': '2024-10-04T19:15:18.187471', 'eventTimesHistory': [], 'questionId': '6dc12d68-af7f-441f-9839-5882e115a2ca', 'questionIdsHistory': ['question_442', 'question_801', 'question_633', 'question_616', 'question_465', 'question_184', 'question_283', 'question_327'], 'skillId': 'sqyudjgp'}, 'parsed_json_length': 353, 'received_content_type': 'application/json', 'received_data_length': 353, 'received_raw_data': '{"skillId": "sqyudjgp", "questionId": "6dc12d68-af7f-441f-9839-5882e115a2ca", "eventTime": "2024-10-04T19:15:18.187471", "questionIdsHistory": ["ques

In [18]:
reproduced_message_2 = {"correctnessHistory": [0, 1, 0, 1, 0.75, 1], 
                        "durationSecondsHistory": [None, None, None, None, None, None], 
                        "eventTime": "2024-12-10T13:34:29.371403527Z", 
                        "eventTimesHistory": ["2024-12-10T13:34:29.371403527Z", "2024-12-10T13:34:20.931262655Z", None, None, None, None], "questionId": "6dc12d68-af7f-441f-9839-5882e115a2ca", 
                        "questionIdsHistory": ["6dc12d68-af7f-441f-9839-5882e115a2ca", "5c686c0c-d332-400e-b621-a8de2d0f824f", None, None, None, None], "skillId": "sqyudjgp"}


print('submitting: ', reproduced_message_2)
response = runtime.invoke_endpoint(EndpointName=endpoint_name, 
                                   ContentType='application/json', 
                                   Body=json.dumps(reproduced_message_2))
result = json.loads(response['Body'].read().decode())

print('result: ', result)

submitting:  {'correctnessHistory': [0, 1, 0, 1, 0.75, 1], 'durationSecondsHistory': [None, None, None, None, None, None], 'eventTime': '2024-12-10T13:34:29.371403527Z', 'eventTimesHistory': ['2024-12-10T13:34:29.371403527Z', '2024-12-10T13:34:20.931262655Z', None, None, None, None], 'questionId': '6dc12d68-af7f-441f-9839-5882e115a2ca', 'questionIdsHistory': ['6dc12d68-af7f-441f-9839-5882e115a2ca', '5c686c0c-d332-400e-b621-a8de2d0f824f', None, None, None, None], 'skillId': 'sqyudjgp'}
result:  {'debug_info': {'parsed_json': {'correctnessHistory': [0, 1, 0, 1, 0.75, 1], 'durationSecondsHistory': [None, None, None, None, None, None], 'eventTime': '2024-12-10T13:34:29.371403527Z', 'eventTimesHistory': ['2024-12-10T13:34:29.371403527Z', '2024-12-10T13:34:20.931262655Z', None, None, None, None], 'questionId': '6dc12d68-af7f-441f-9839-5882e115a2ca', 'questionIdsHistory': ['6dc12d68-af7f-441f-9839-5882e115a2ca', '5c686c0c-d332-400e-b621-a8de2d0f824f', None, None, None, None], 'skillId': 'sqyu

In [19]:
result['prediction']

{'item_prediction': [0.5081777572631836],
 'item_prediction_confidence': [81.75720992622401],
 'item_prediction_error': [0.4648544490337372],
 'model_version': ['4.0a5'],
 'skill_prediction': [0.21159693598747253],
 'skill_prediction_confidence': [82.71044707964097],
 'skill_prediction_error': [0.3843223750591278]}

In [20]:
test_message_multiple_events = [{'skillId': '5111f61e-e69f-e311-9503-005056801da1',
  'questionId': 'question_136',
  'eventTime': '2024-10-04T19:15:18.187471',
  'questionIdsHistory': ['question_852',
   'question_204',
   'question_75',
   'question_862',
   'question_775',
   'question_649',
   'question_928',
   'question_242',
   'question_322'],
  'correctnessHistory': [100, 100, 100, 100, 0, 0, 0, 0, 0],
  'durationSecondsHistory': [None, None,None,None,None,None,None,None,None],
  'eventTimesHistory': []},
{'skillId': '5111f61e-e69f-e311-9503-005056801da1',
  'questionId': 'question_136',
  'eventTime': '2024-10-04T19:15:18.187471',
  'questionIdsHistory': ['question_852',
   'question_204',
   'question_75',
   'question_862',
   'question_775',
   'question_649',
   'question_928',
   'question_242',
   'question_322'],
  'correctnessHistory': [0, 0, 0, 0, 0, 100, 100, 100, 100],
  'durationSecondsHistory': [None, None,None,None,None,None,None,None,None],
  'eventTimesHistory': []},
{'skillId': '5111f61e-e69f-e311-9503-005056801da1',
  'questionId': 'question_136',
  'eventTime': '2024-10-04T19:15:18.187471',
  'questionIdsHistory': ['question_852',
   'question_204',
   'question_75',
   'question_862',
   'question_775',
   'question_649',
   'question_928',
   'question_242',
   'question_322'],
  'correctnessHistory': [0, 0, 100, 100, 100, 100, 0, 0, 0],
  'durationSecondsHistory': [None, None,None,None,None,None,None,None,None],
  'eventTimesHistory': []},
{'skillId': '5111f61e-e69f-e311-9503-005056801da1',
  'questionId': 'question_136',
  'eventTime': '2024-10-04T19:15:18.187471',
  'questionIdsHistory': ['question_852',
   'question_204',
   'question_75',
   'question_862',
   'question_775',
   'question_649',
   'question_928',
   'question_242',
   'question_322'],
  'correctnessHistory': [100, 100, 0, 0, 0, 0, 0, 100, 100],
  'durationSecondsHistory': [None, None,None,None,None,None,None,None,None],
  'eventTimesHistory': []}]


#test endpoint
import boto3
import json

runtime = boto3.client('sagemaker-runtime')
print('submitting: ', test_message_multiple_events)
response = runtime.invoke_endpoint(EndpointName=endpoint_name, 
                                   ContentType='application/json', 
                                   Body=json.dumps(test_message_multiple_events))
result = json.loads(response['Body'].read().decode())

# print('result: ', result)
result['prediction']

submitting:  [{'skillId': '5111f61e-e69f-e311-9503-005056801da1', 'questionId': 'question_136', 'eventTime': '2024-10-04T19:15:18.187471', 'questionIdsHistory': ['question_852', 'question_204', 'question_75', 'question_862', 'question_775', 'question_649', 'question_928', 'question_242', 'question_322'], 'correctnessHistory': [100, 100, 100, 100, 0, 0, 0, 0, 0], 'durationSecondsHistory': [None, None, None, None, None, None, None, None, None], 'eventTimesHistory': []}, {'skillId': '5111f61e-e69f-e311-9503-005056801da1', 'questionId': 'question_136', 'eventTime': '2024-10-04T19:15:18.187471', 'questionIdsHistory': ['question_852', 'question_204', 'question_75', 'question_862', 'question_775', 'question_649', 'question_928', 'question_242', 'question_322'], 'correctnessHistory': [0, 0, 0, 0, 0, 100, 100, 100, 100], 'durationSecondsHistory': [None, None, None, None, None, None, None, None, None], 'eventTimesHistory': []}, {'skillId': '5111f61e-e69f-e311-9503-005056801da1', 'questionId': 'q

{'item_prediction': [0.6732555627822876,
  0.5121054649353027,
  0.6504027843475342,
  0.5483493804931641],
 'item_prediction_confidence': [82.71044707964097,
  49.11454542859997,
  53.581060833083605,
  2.1176705623813805],
 'item_prediction_error': [0.43329405784606934,
  0.47884616255760193,
  0.4548545777797699,
  0.4738670587539673],
 'model_version': ['4.0a5', '4.0a5', '4.0a5', '4.0a5'],
 'skill_prediction': [0.6856043934822083,
  0.4637630879878998,
  0.5047964453697205,
  0.591255247592926],
 'skill_prediction_confidence': [82.71044707964097,
  49.11454542859997,
  13.158382921643335,
  18.64663156242419],
 'skill_prediction_error': [0.4147034287452698,
  0.4780576825141907,
  0.4855242371559143,
  0.47181394696235657]}

In [21]:
inf_test =  test_input = {
  "skillId": "evnldqti",
  "questionId": "94046f4d-7f58-4ed6-ab60-ca6edfc300f4",
  "eventTime": "2024-06-05 07:03:58.636000+00:00",
  "questionIdsHistory": [
    "94046f4d-7f58-4ed6-ab60-ca6edfc300f4",
    "fe63aeb7-9d8e-43a5-857b-6ce7d4a38a63",
    "f8187d97-6abf-4d5d-8556-eb4416126031",
    "e3f28d4d-43f1-475e-9ecf-b961aac28ed7",
    "d330d5a4-b374-47f3-a40a-1180518cbe8f",
    "cdd3cff8-3fa0-4a0f-a72e-f0272702950d",
    "b22aba47-c2a5-4868-8e52-6be05eb7732a",
    "af90d47d-7a56-4653-90b1-3791233f7be3",
    "9dc1166a-f460-41ec-8dbb-44b1f7b8e477",
    "888949ce-ca06-452d-8813-3275b51212d5",
    "7363d704-a28e-417b-b6bc-840864f791ef"
  ],
  "correctnessHistory": [
    0,
    100,
    100,
    100,
    100,
    0,
    0,
    100,
    100,
    100,
    0
  ],
  "durationSecondsHistory": [],
  "eventTimesHistory": [
    "2024-06-05T07:03:58.000Z",
    "2024-05-15T07:10:32.000Z",
    "2024-05-15T07:10:27.000Z",
    "2024-05-15T07:10:23.000Z",
    "2024-05-15T07:10:19.000Z",
    "2024-05-15T07:10:08.000Z",
    "2024-05-15T07:09:49.000Z",
    "2024-05-15T07:09:40.000Z",
    "2024-05-15T07:09:34.000Z",
    "2024-05-15T07:09:31.000Z",
    "2024-05-15T07:07:57.000Z"
  ]
}


import boto3
runtime = boto3.client('sagemaker-runtime')
print('submitting: ', inf_test)
response = runtime.invoke_endpoint(EndpointName=endpoint_name, 
                                   ContentType='application/json', 
                                   Body=json.dumps(inf_test))
result = json.loads(response['Body'].read().decode())

print('result: ', result)
result['prediction']

submitting:  {'skillId': 'evnldqti', 'questionId': '94046f4d-7f58-4ed6-ab60-ca6edfc300f4', 'eventTime': '2024-06-05 07:03:58.636000+00:00', 'questionIdsHistory': ['94046f4d-7f58-4ed6-ab60-ca6edfc300f4', 'fe63aeb7-9d8e-43a5-857b-6ce7d4a38a63', 'f8187d97-6abf-4d5d-8556-eb4416126031', 'e3f28d4d-43f1-475e-9ecf-b961aac28ed7', 'd330d5a4-b374-47f3-a40a-1180518cbe8f', 'cdd3cff8-3fa0-4a0f-a72e-f0272702950d', 'b22aba47-c2a5-4868-8e52-6be05eb7732a', 'af90d47d-7a56-4653-90b1-3791233f7be3', '9dc1166a-f460-41ec-8dbb-44b1f7b8e477', '888949ce-ca06-452d-8813-3275b51212d5', '7363d704-a28e-417b-b6bc-840864f791ef'], 'correctnessHistory': [0, 100, 100, 100, 100, 0, 0, 100, 100, 100, 0], 'durationSecondsHistory': [], 'eventTimesHistory': ['2024-06-05T07:03:58.000Z', '2024-05-15T07:10:32.000Z', '2024-05-15T07:10:27.000Z', '2024-05-15T07:10:23.000Z', '2024-05-15T07:10:19.000Z', '2024-05-15T07:10:08.000Z', '2024-05-15T07:09:49.000Z', '2024-05-15T07:09:40.000Z', '2024-05-15T07:09:34.000Z', '2024-05-15T07:09:31.

{'item_prediction': [0.5968288779258728],
 'item_prediction_confidence': [56.35372518800747],
 'item_prediction_error': [0.4941800832748413],
 'model_version': ['4.0a5'],
 'skill_prediction': [0.41462963819503784],
 'skill_prediction_confidence': [0.0014270017266720893],
 'skill_prediction_error': [0.5100198984146118]}

In [22]:
# CHECK FOR NANs because python won't error out but Haskell will

In [23]:
def check_for_nans(response_body):
    return 'nan' in str(response_body)

# Use it after your API call:
response = runtime.invoke_endpoint(EndpointName=endpoint_name, 
                                   ContentType='application/json', 
                                   Body=json.dumps(inf_test))
result = json.loads(response['Body'].read().decode())
assert not check_for_nans(result), "Found NaN values in response - will cause Haskell server error"

Cloudwatch metrics, instance counts, and scaling progress

In [24]:
import boto3
from datetime import datetime, timedelta

cloudwatch = boto3.client('cloudwatch')

response = cloudwatch.get_metric_statistics(
    Namespace='AWS/SageMaker',
    MetricName='Invocations',
    Dimensions=[{'Name': 'EndpointName', 'Value': endpoint_name}],
    StartTime=datetime.utcnow() - timedelta(hours=24),
    EndTime=datetime.utcnow(),
    Period=60,
    Statistics=['Sum']
)
response

{'Label': 'Invocations',
 'Datapoints': [],
 'ResponseMetadata': {'RequestId': '793dc34d-9c85-44a9-82db-f9bf102b72c5',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '793dc34d-9c85-44a9-82db-f9bf102b72c5',
   'content-type': 'text/xml',
   'content-length': '334',
   'date': 'Sun, 19 Jan 2025 21:28:58 GMT'},
  'RetryAttempts': 0}}

In [25]:
import boto3

client = boto3.client('sagemaker')

response = client.describe_endpoint(
    EndpointName='[REDACTED]'
)

print('scaling policy:')
print(json.dumps(response, indent=2, default=str))

app_scaling = boto3.client('application-autoscaling')

try:
    response = app_scaling.describe_scaling_policies(
        ServiceNamespace='sagemaker',
        ResourceId=f'endpoint/[REDACTED]/variant/AllTraffic'
    )
    print('\n\ncurrent instance count/desired instance: ')
    print(json.dumps(response, indent=2, default=str))
except Exception as e:
    print("Error:", e)
    
    

scaling policy:
{
  "EndpointName": "[REDACTED]",
  "EndpointArn": "arn:aws:sagemaker:us-east-1:[REDACTED]:endpoint/[REDACTED]",
  "EndpointConfigName": "ela-20250119212457",
  "ProductionVariants": [
    {
      "VariantName": "AllTraffic",
      "DeployedImages": [
        {
          "SpecifiedImage": "[REDACTED].dkr.ecr.us-east-1.amazonaws.com/[REDACTED]-ml/ela-proficiency-model:1737321886461",
          "ResolvedImage": "[REDACTED].dkr.ecr.us-east-1.amazonaws.com/[REDACTED]-ml/ela-proficiency-model@sha256:7a7b5403128c23d21b4519880fdc8cc655b2fea79602edced1e595728c3525d9",
          "ResolutionTime": "2025-01-19 21:24:58.891000+00:00"
        }
      ],
      "CurrentWeight": 1.0,
      "DesiredWeight": 1.0,
      "CurrentInstanceCount": 1,
      "DesiredInstanceCount": 1
    }
  ],
  "EndpointStatus": "InService",
  "CreationTime": "2024-10-04 19:24:47.157000+00:00",
  "LastModifiedTime": "2025-01-19 21:28:09.040000+00:00",
  "ResponseMetadata": {
    "RequestId": "3e94a724-1b59-4d

In [26]:
#ADJUST PARAMTERS AT TOP OF NOTEBOOK TO MAINTAIN CONSISTENCY, 
# FUNCTION BELOW REQUIRES VARIABLES ABOVE
# update_endpoint_with_scaling(
#     endpoint_name=endpoint_name,
#     new_image_uri=image_uri,
#     test_message=test_message,
#     min_capacity=min_capacity,
#     max_capacity=max_capacity,
#     target_value=target_value)

In [27]:
# setup_auto_scaling(endpoint_name, min_capacity, max_capacity, target_value, 
#                        scale_in_cooldown, scale_out_cooldown)

In [28]:
verify_scaling_policy(endpoint_name)

Current endpoint configuration:
Instance Type: ml.c5.large
Min Capacity: 1
Max Capacity: 100

Scaling policy configuration:
Policy Name: ScalingPolicy-[REDACTED]
Target Value: 15.0
Scale-in Cooldown: 60 seconds
Scale-out Cooldown: 300 seconds


In [29]:
check_scaling_activities(endpoint_name, MaxResults=500)

Recent scaling activities:

Activity ID: 152f14a6-2b14-4092-b0ac-964cd6be65a6
Status: Successful
Description: Setting desired instance count to 5.
Cause: monitor alarm TargetTracking-endpoint/[REDACTED]/variant/AllTraffic-AlarmHigh-145e263a-c609-4b6d-892f-b8b94f885fb6 in state ALARM triggered policy ScalingPolicy-[REDACTED]
Start Time: 2025-01-19 19:56:33.406000+00:00
Status Message: Successfully set desired instance count to 5. Change successfully fulfilled by sagemaker.

Activity ID: ab7ff491-0037-4ff2-978e-85cc93cac42f
Status: Successful
Description: Setting desired instance count to 4.
Cause: monitor alarm TargetTracking-endpoint/[REDACTED]/variant/AllTraffic-AlarmHigh-145e263a-c609-4b6d-892f-b8b94f885fb6 in state ALARM triggered policy ScalingPolicy-[REDACTED]
Start Time: 2025-01-19 19:52:33.436000+00:00
Status Message: Successfully set desired instance count to 4. Change successfully fulfilled by sagemaker.

Activity ID: 366cab55-7db4-4654-92f4-8d4d371453a0
Status: Successful
Des